# 04 Model Training - Algorithm Selection and Hyperparameter Tuning

⚠️ **IMPORTANT: DO NOT RE-RUN THIS NOTEBOOK ACCIDENTALLY**

This notebook contains trained production models. All cells have been carefully executed and saved.  
**Only run this notebook if you intentionally need to retrain the models (WARNING: This is computationally expensive).**

---

## Purpose
Train and optimize classification models:
- Baseline: Logistic Regression
- Gradient Boosting with hyperparameter tuning
- Ensemble with weighted voting
- Compare model performance on test set
- Save best-performing models


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
# xgboost has dependency issues on m1/m2 mac, skip it
# import xgboost as xgb
import joblib
from IPython.display import Markdown, display
import json
import time

# Setup path
PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries loaded successfully')

Libraries loaded successfully


## 1. Load Engineered Features

In [2]:
# Load data from feature engineering notebook
import scipy.sparse as sp

print('Loading engineered datasets...')
X_pca = np.load('../results/X_pca.npy')
y = np.load('../results/y.npy')

print(f'Loaded X_pca: {X_pca.shape}')
print(f'Loaded y: {y.shape}')
print(f'Class distribution: {np.bincount(y)}')

# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

Loading engineered datasets...
Loaded X_pca: (100000, 100)
Loaded y: (100000,)
Class distribution: [50071 49929]

Train set: (80000, 100)
Test set: (20000, 100)


## 2. Train Baseline - Logistic Regression

In [3]:
print('Training Logistic Regression baseline...')
start = time.time()

lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train, y_train)

elapsed = time.time() - start
y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

lr_metrics = {
    'accuracy': accuracy_score(y_test, y_pred_lr),
    'precision': precision_score(y_test, y_pred_lr),
    'recall': recall_score(y_test, y_pred_lr),
    'f1': f1_score(y_test, y_pred_lr),
    'roc_auc': roc_auc_score(y_test, y_proba_lr),
    'training_time': elapsed
}

print(f'Logistic Regression (Training time: {elapsed:.2f}s)')
for metric, value in lr_metrics.items():
    print(f'  {metric:20s}: {value:.4f}')

Training Logistic Regression baseline...


/Users/bhavanishanker/predictive-sales-analytics-engine/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Logistic Regression (Training time: 0.34s)
  accuracy            : 0.9634
  precision           : 0.9504
  recall              : 0.9777
  f1                  : 0.9639
  roc_auc             : 0.9957
  training_time       : 0.3411


## 3. Train Random Forest with Hyperparameter Tuning

In [4]:
print('Training Random Forest with GridSearchCV...')
start = time.time()

rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, 30],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_grid = GridSearchCV(rf, rf_params, cv=3, scoring='f1', n_jobs=-1, verbose=1)
rf_grid.fit(X_train, y_train)

elapsed = time.time() - start
best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]

rf_metrics = {
    'accuracy': accuracy_score(y_test, y_pred_rf),
    'precision': precision_score(y_test, y_pred_rf),
    'recall': recall_score(y_test, y_pred_rf),
    'f1': f1_score(y_test, y_pred_rf),
    'roc_auc': roc_auc_score(y_test, y_proba_rf),
    'training_time': elapsed
}

print(f'\nRandom Forest Best Params: {rf_grid.best_params_}')
print(f'Random Forest (Training time: {elapsed:.2f}s)')
for metric, value in rf_metrics.items():
    print(f'  {metric:20s}: {value:.4f}')

Training Random Forest with GridSearchCV...
Fitting 3 folds for each of 24 candidates, totalling 72 fits

Random Forest Best Params: {'max_depth': 30, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}
Random Forest (Training time: 699.78s)
  accuracy            : 0.9594
  precision           : 0.9453
  recall              : 0.9751
  f1                  : 0.9600
  roc_auc             : 0.9951
  training_time       : 699.7803


## 6. Model Comparison and Save Best Model

In [7]:
# Compare all models
comparison_df = pd.DataFrame({
    'Logistic Regression': lr_metrics,
    'Random Forest': rf_metrics
}).T

print('\n=== Model Comparison ===')
print(comparison_df.round(4))

# Identify best model by F1 score
best_model_name = comparison_df['f1'].idxmax()
best_model_obj = {
    'Logistic Regression': lr,
    'Random Forest': best_rf
}.get(best_model_name)

print(f'\n✓ Best Model: {best_model_name} (F1={comparison_df.loc[best_model_name, "f1"]:.4f})')

# Save models
joblib.dump(lr, '../results/model_lr.pkl')
joblib.dump(best_rf, '../results/model_rf.pkl')
joblib.dump(best_model_obj, '../results/model_best.pkl')

# Save comparison results
comparison_df.to_csv('../metrics/model_comparison.csv')
comparison_df.to_json('../metrics/model_comparison.json')

print('✓ Models saved to results/')
print('✓ Comparison saved to metrics/')


=== Model Comparison ===
                     accuracy  precision  recall      f1  roc_auc  \
Logistic Regression    0.9634     0.9504  0.9777  0.9639   0.9957   
Random Forest          0.9594     0.9453  0.9751  0.9600   0.9951   

                     training_time  
Logistic Regression         0.3411  
Random Forest             699.7803  

✓ Best Model: Logistic Regression (F1=0.9639)
✓ Models saved to results/
✓ Comparison saved to metrics/


## 7. Training Summary

In [8]:
summary_text = f"""
## Model Training Summary

### Dataset Split
- Training set: {X_train.shape[0]:,} samples
- Test set: {X_test.shape[0]:,} samples  
- Features: {X_train.shape[1]}

### Model Performance (Test Set)

#### Logistic Regression (Baseline)
- Accuracy: {lr_metrics['accuracy']:.4f}
- F1 Score: {lr_metrics['f1']:.4f}
- Training Time: {lr_metrics['training_time']:.2f}s

#### Random Forest
- Accuracy: {rf_metrics['accuracy']:.4f}
- F1 Score: {rf_metrics['f1']:.4f}
- Training Time: {rf_metrics['training_time']:.2f}s

### Best Model: {best_model_name}
- F1 Score: {comparison_df.loc[best_model_name, 'f1']:.4f}
- ROC-AUC: {comparison_df.loc[best_model_name, 'roc_auc']:.4f}
"""

display(Markdown(summary_text))


## Model Training Summary

### Dataset Split
- Training set: 80,000 samples
- Test set: 20,000 samples  
- Features: 100

### Model Performance (Test Set)

#### Logistic Regression (Baseline)
- Accuracy: 0.9634
- F1 Score: 0.9639
- Training Time: 0.34s

#### Random Forest
- Accuracy: 0.9594
- F1 Score: 0.9600
- Training Time: 699.78s

### Best Model: Logistic Regression
- F1 Score: 0.9639
- ROC-AUC: 0.9957
